# Question 1
Implement the Minimum Edit Distance algorithm to find the edit distance between
any two given strings.
Also, list the edit operations.

In [ ]:
def edit_distance(s1, s2):

    m = len(s1)
    n = len(s2)

    # Create DP table
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    # If s2 is empty, delete all characters from s1
    for i in range(m + 1):
        dp[i][0] = i

    # If s1 is empty, insert all characters of s2
    for j in range(n + 1):
        dp[0][j] = j

    # Fill the table
    for i in range(1, m + 1):
        for j in range(1, n + 1):

            if s1[i - 1] == s2[j - 1]:
                # Characters are same
                dp[i][j] = dp[i - 1][j - 1]

            else:
                # Choose minimum of:
                # Delete, Insert, Replace
                dp[i][j] = 1 + min(
                    dp[i - 1][j],      # Delete
                    dp[i][j - 1],      # Insert
                    dp[i - 1][j - 1]   # Replace
                )

    return dp[m][n]


# Example
s1 = input("Enter first string: ")
s2 = input("Enter second string: ")

distance = edit_distance(s1, s2)

print("Minimum Edit Distance:", distance)

Enter first string: cat
Enter second string: cut
Minimum Edit Distance: 1


# Question 2
Design and implement a statistical spell checker for detecting and correcting
non-word spelling errors in English, using the bigram language model. Your
program should do the following:

1. Tokenize the corpus and create a vocabulary of unique words.
2. Create a bi-gram frequency table for all possible bigrams in the corpus.
3. Scan the given input text to identify the non-word spelling errors
4. Generate the candidate list using 1 edit distance from the misspelled words
5. Suggest the best candidate word by calculating the probability of the given
sentence using the bigram LM.



In [ ]:
import re
from collections import Counter


# --------------------------------------------------
# 1. TOKENIZE CORPUS AND CREATE VOCABULARY
# --------------------------------------------------

corpus = """
the cat sat on the mat
the cat ate the fish
the dog sat on the mat
the dog ate the food
the fish was on the mat
"""

# Get words from corpus
words = re.findall(r"[a-z]+", corpus.lower())

# Unique vocabulary
vocabulary = set(words)

print("Vocabulary:")
print(vocabulary)


# --------------------------------------------------
# 2. CREATE BIGRAM FREQUENCY TABLE
# --------------------------------------------------

bigram_counts = Counter()

for i in range(len(words) - 1):
    bigram = (words[i], words[i + 1])
    bigram_counts[bigram] += 1

print("\nBigram Frequencies:")
for bigram, count in bigram_counts.items():
    print(bigram, ":", count)


# Count how many times each word occurs
word_counts = Counter(words)


# --------------------------------------------------
# BIGRAM PROBABILITY
# P(word2 | word1)
# --------------------------------------------------

def bigram_probability(word1, word2):

    # Add-one smoothing
    numerator = bigram_counts[(word1, word2)] + 1
    denominator = word_counts[word1] + len(vocabulary)

    return numerator / denominator


# --------------------------------------------------
# 3. FIND EDIT DISTANCE 1 CANDIDATES
# --------------------------------------------------

def edit_distance_one(word):

    letters = "abcdefghijklmnopqrstuvwxyz"
    candidates = set()

    # DELETE one character
    for i in range(len(word)):
        candidate = word[:i] + word[i + 1:]
        candidates.add(candidate)

    # INSERT one character
    for i in range(len(word) + 1):
        for c in letters:
            candidate = word[:i] + c + word[i:]
            candidates.add(candidate)

    # REPLACE one character
    for i in range(len(word)):
        for c in letters:
            candidate = word[:i] + c + word[i + 1:]
            candidates.add(candidate)

    return candidates


# --------------------------------------------------
# 4. FIND BEST SPELLING CANDIDATE
# --------------------------------------------------

def correct_word(previous_word, misspelled_word):

    candidates = edit_distance_one(misspelled_word)

    # Only keep candidates that exist in vocabulary
    candidates = candidates.intersection(vocabulary)

    if not candidates:
        return misspelled_word

    # Choose candidate with highest bigram probability
    best_candidate = misspelled_word
    best_probability = 0

    for candidate in candidates:

        probability = bigram_probability(
            previous_word,
            candidate
        )

        if probability > best_probability:
            best_probability = probability
            best_candidate = candidate

    return best_candidate


# --------------------------------------------------
# 5. SCAN INPUT TEXT
# --------------------------------------------------

text = input("\nEnter a sentence: ")

input_words = re.findall(r"[a-z]+", text.lower())

corrected_words = []

previous_word = None

for word in input_words:

    # Word is correct
    if word in vocabulary:

        corrected_words.append(word)
        previous_word = word

    # Word is not in vocabulary
    else:

        if previous_word is not None:

            correction = correct_word(
                previous_word,
                word
            )

        else:
            # No previous word, choose any valid candidate
            candidates = edit_distance_one(word)
            candidates = candidates.intersection(vocabulary)

            if candidates:
                correction = max(
                    candidates,
                    key=lambda x: word_counts[x]
                )
            else:
                correction = word

        print(f"\nMisspelled word: {word}")
        print("Candidates:", edit_distance_one(word).intersection(vocabulary))
        print("Suggested correction:", correction)

        corrected_words.append(correction)
        previous_word = correction


# --------------------------------------------------
# FINAL SENTENCE
# --------------------------------------------------

print("\nCorrected sentence:")
print(" ".join(corrected_words))

Vocabulary:
{'cat', 'sat', 'fish', 'food', 'the', 'was', 'on', 'dog', 'ate', 'mat'}

Bigram Frequencies:
('the', 'cat') : 2
('cat', 'sat') : 1
('sat', 'on') : 2
('on', 'the') : 3
('the', 'mat') : 3
('mat', 'the') : 2
('cat', 'ate') : 1
('ate', 'the') : 2
('the', 'fish') : 2
('fish', 'the') : 1
('the', 'dog') : 2
('dog', 'sat') : 1
('dog', 'ate') : 1
('the', 'food') : 1
('food', 'the') : 1
('fish', 'was') : 1
('was', 'on') : 1

Enter a sentence: cat ato fish

Misspelled word: ato
Candidates: {'ate'}
Suggested correction: ate

Corrected sentence:
cat ate fish


# Question 3
Implement a noisy channel spell checker for non-word spelling errors in the English language, using the same dataset.

Compare its performance with the bigram LM spellchecker from the previous question.

In [ ]:
import re
from collections import Counter


# --------------------------------------------------
# 1. CORPUS
# --------------------------------------------------

corpus = """
the cat sat on the mat
the cat ate the fish
the dog sat on the mat
the dog ate the food
the fish was on the mat
"""

# Tokenize corpus
words = re.findall(r"[a-z]+", corpus.lower())

# Vocabulary
vocabulary = set(words)

# Word frequencies
word_counts = Counter(words)


# --------------------------------------------------
# 2. BIGRAM FREQUENCY TABLE
# --------------------------------------------------

bigram_counts = Counter()

for i in range(len(words) - 1):
    bigram = (words[i], words[i + 1])
    bigram_counts[bigram] += 1


# --------------------------------------------------
# 3. BIGRAM PROBABILITY
# --------------------------------------------------

def bigram_probability(previous, word):

    numerator = bigram_counts[(previous, word)] + 1
    denominator = word_counts[previous] + len(vocabulary)

    return numerator / denominator


# --------------------------------------------------
# 4. EDIT DISTANCE
# --------------------------------------------------

def edit_distance(s1, s2):

    m = len(s1)
    n = len(s2)

    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i

    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):

            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]

            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],      # delete
                    dp[i][j - 1],      # insert
                    dp[i - 1][j - 1]   # replace
                )

    return dp[m][n]


# --------------------------------------------------
# 5. GENERATE CANDIDATES
# --------------------------------------------------

def generate_candidates(word):

    candidates = []

    for candidate in vocabulary:

        # Only consider words within edit distance 1
        if edit_distance(word, candidate) == 1:
            candidates.append(candidate)

    return candidates


# --------------------------------------------------
# 6. NOISY CHANNEL PROBABILITY
# --------------------------------------------------

def noisy_channel_probability(wrong_word, candidate):

    distance = edit_distance(wrong_word, candidate)

    # Simple assumption:
    # smaller edit distance = more likely typo

    if distance == 1:
        return 1.0

    elif distance == 2:
        return 0.5

    else:
        return 0.1


# --------------------------------------------------
# 7. NOISY CHANNEL SPELL CHECKER
# --------------------------------------------------

def noisy_channel_correct(previous_word, wrong_word):

    candidates = generate_candidates(wrong_word)

    if not candidates:
        return wrong_word

    best_candidate = wrong_word
    best_score = 0

    for candidate in candidates:

        # P(wrong word | candidate)
        channel_probability = noisy_channel_probability(
            wrong_word,
            candidate
        )

        # P(candidate | previous word)
        language_probability = bigram_probability(
            previous_word,
            candidate
        )

        # Noisy Channel Model
        score = channel_probability * language_probability

        if score > best_score:
            best_score = score
            best_candidate = candidate

    return best_candidate


# --------------------------------------------------
# 8. BIGRAM-ONLY SPELL CHECKER
# --------------------------------------------------

def bigram_correct(previous_word, wrong_word):

    candidates = generate_candidates(wrong_word)

    if not candidates:
        return wrong_word

    best_candidate = wrong_word
    best_probability = 0

    for candidate in candidates:

        probability = bigram_probability(
            previous_word,
            candidate
        )

        if probability > best_probability:
            best_probability = probability
            best_candidate = candidate

    return best_candidate


# --------------------------------------------------
# 9. SPELL CHECK INPUT
# --------------------------------------------------

def spell_check(text, method):

    input_words = re.findall(r"[a-z]+", text.lower())

    corrected = []

    previous_word = None

    for word in input_words:

        if word in vocabulary:

            corrected.append(word)
            previous_word = word

        else:

            if previous_word is not None:

                if method == "bigram":
                    correction = bigram_correct(
                        previous_word,
                        word
                    )

                else:
                    correction = noisy_channel_correct(
                        previous_word,
                        word
                    )

            else:
                correction = word

            corrected.append(correction)
            previous_word = correction

    return " ".join(corrected)


# --------------------------------------------------
# 10. MAIN PROGRAM
# --------------------------------------------------

text = input("Enter a sentence: ")

bigram_result = spell_check(text, "bigram")
noisy_result = spell_check(text, "noisy")

print("\nOriginal:")
print(text)

print("\nBigram LM Correction:")
print(bigram_result)

print("\nNoisy Channel Correction:")
print(noisy_result)

Enter a sentence: the cat mat

Original:
the cat mat

Bigram LM Correction:
the cat mat

Noisy Channel Correction:
the cat mat


Design and implement a Binary Logistic Regression classifier to detect and correct real-word homophone errors in input text using the provided confusion sets:

Your system must support the following confusion sets: {"write", "right", "rite"},: {"peace", "piece"}, {"their", "there", "they're"}.

Your program should do the following:

*   Scan input tokens. If a token belongs to any supported confusion set, trigger candidate evaluation.
* Extract features for all candidates in that specific confusion set.
* Use your trained Logistic Regression model to predict P(y=1| candidate, context) for each candidate.
* Replace the target word with the highest-scoring candidate.

*Note: Because different confusion sets rely on different linguistic signals, your features must generalize across all three sets. Design at least 3 distinct feature types (e.g., n-gram log-probabilities, unigram priors, syntax/POS indicators, exact-match flags).*

In [ ]:
import re
import math
from collections import Counter
from sklearn.linear_model import LogisticRegression


# ============================================================
# 1. CONFUSION SETS
# ============================================================

CONFUSION_SETS = [
    {"write", "right", "rite"},
    {"peace", "piece"},
    {"their", "there", "they're"}
]


# ============================================================
# 2. TRAINING DATA
#    Each sentence has the CORRECT word marked.
# ============================================================

TRAIN_DATA = [

    # write / right / rite
    ("I want to write a letter", "write"),
    ("Please write your name", "write"),
    ("She will write a story", "write"),
    ("He can write very well", "write"),

    ("Turn right at the corner", "right"),
    ("Take a right turn", "right"),
    ("You are right about this", "right"),
    ("The shop is on the right", "right"),

    ("The wedding rite was beautiful", "rite"),
    ("They performed the ancient rite", "rite"),

    # peace / piece
    ("We all want peace", "peace"),
    ("The country needs peace", "peace"),
    ("They finally made peace", "peace"),

    ("I ate a piece of cake", "piece"),
    ("Give me a piece of paper", "piece"),
    ("She broke a piece of glass", "piece"),

    # their / there / they're
    ("Their house is beautiful", "their"),
    ("I like their new car", "their"),
    ("Their dog is friendly", "their"),

    ("The book is there", "there"),
    ("Put the bag there", "there"),
    ("There is a problem", "there"),

    ("They're going to school", "they're"),
    ("They're very happy", "they're"),
    ("I think they're coming", "they're"),
]


# ============================================================
# 3. BUILD WORD AND BIGRAM COUNTS
# ============================================================

all_words = []

for sentence, correct_word in TRAIN_DATA:
    words = re.findall(r"[a-z']+", sentence.lower())
    all_words.extend(words)

unigram_counts = Counter(all_words)

bigram_counts = Counter()

for sentence, correct_word in TRAIN_DATA:
    words = re.findall(r"[a-z']+", sentence.lower())

    for i in range(len(words) - 1):
        bigram_counts[(words[i], words[i + 1])] += 1

vocabulary_size = len(unigram_counts)


# ============================================================
# 4. PROBABILITY FUNCTIONS
# ============================================================

def unigram_probability(word):
    """
    P(word)
    Add-one smoothing is used.
    """
    return (
        unigram_counts[word] + 1
    ) / (
        sum(unigram_counts.values()) + vocabulary_size
    )


def bigram_probability(word1, word2):
    """
    P(word2 | word1)
    """
    return (
        bigram_counts[(word1, word2)] + 1
    ) / (
        unigram_counts[word1] + vocabulary_size
    )


# ============================================================
# 5. FEATURE EXTRACTION
# ============================================================

def extract_features(candidate, previous_word, next_word,
                     original_word):

    # Feature 1: Unigram log probability
    unigram_feature = math.log(
        unigram_probability(candidate)
    )

    # Feature 2: Left bigram probability
    left_bigram_feature = math.log(
        bigram_probability(previous_word, candidate)
    )

    # Feature 3: Right bigram probability
    right_bigram_feature = math.log(
        bigram_probability(candidate, next_word)
    )

    # Feature 4: Exact match flag
    exact_match = int(candidate == original_word)

    return [
        unigram_feature,
        left_bigram_feature,
        right_bigram_feature,
        exact_match
    ]


# ============================================================
# 6. CREATE TRAINING FEATURES
# ============================================================

X = []
y = []

for sentence, correct_word in TRAIN_DATA:

    words = re.findall(r"[a-z']+", sentence.lower())

    for i, word in enumerate(words):

        # Only train on confusion-set words
        confusion_set = None

        for group in CONFUSION_SETS:
            if word in group:
                confusion_set = group
                break

        if confusion_set is None:
            continue

        # Get context
        previous_word = words[i - 1] if i > 0 else "<START>"
        next_word = words[i + 1] if i < len(words) - 1 else "<END>"

        # Evaluate every candidate
        for candidate in confusion_set:

            features = extract_features(
                candidate,
                previous_word,
                next_word,
                word
            )

            X.append(features)

            # 1 = correct candidate
            # 0 = incorrect candidate
            y.append(int(candidate == correct_word))


# ============================================================
# 7. TRAIN LOGISTIC REGRESSION
# ============================================================

model = LogisticRegression()

model.fit(X, y)

print("Model trained successfully!")


# ============================================================
# 8. FIND CONFUSION SET FOR A WORD
# ============================================================

def get_confusion_set(word):

    for group in CONFUSION_SETS:
        if word in group:
            return group

    return None


# ============================================================
# 9. CORRECT A WORD
# ============================================================

def correct_word(words, index):

    original_word = words[index]

    confusion_set = get_confusion_set(original_word)

    if confusion_set is None:
        return original_word

    previous_word = (
        words[index - 1]
        if index > 0
        else "<START>"
    )

    next_word = (
        words[index + 1]
        if index < len(words) - 1
        else "<END>"
    )

    candidates = list(confusion_set)

    feature_list = []

    for candidate in candidates:

        features = extract_features(
            candidate,
            previous_word,
            next_word,
            original_word
        )

        feature_list.append(features)

    # Logistic Regression probabilities
    probabilities = model.predict_proba(feature_list)[:, 1]

    # Find candidate with highest P(y=1)
    best_index = probabilities.argmax()

    best_candidate = candidates[best_index]

    print("\nTarget:", original_word)

    for candidate, probability in zip(
        candidates, probabilities
    ):
        print(
            f"{candidate:8} -> "
            f"{probability:.4f}"
        )

    print("Selected:", best_candidate)

    return best_candidate


# ============================================================
# 10. PROCESS INPUT SENTENCE
# ============================================================

def correct_sentence(sentence):

    words = re.findall(r"[a-z']+", sentence.lower())

    corrected_words = []

    for i, word in enumerate(words):

        if get_confusion_set(word):

            corrected = correct_word(words, i)

        else:
            corrected = word

        corrected_words.append(corrected)

    return " ".join(corrected_words)


# ============================================================
# 11. MAIN PROGRAM
# ============================================================

text = input("Enter a sentence: ")

result = correct_sentence(text)

print("\nOriginal:")
print(text)

print("\nCorrected:")
print(result)

Model trained successfully!
Enter a sentence: I want to right a letter

Target: right
right    -> 0.6155
write    -> 0.4659
rite     -> 0.0698
Selected: right

Original:
I want to right a letter

Corrected:
i want to right a letter


Explore any available neural spelling error detection and correction model.

How does the performance of your previous models compare with the neural model evaluated on your chosen dataset?

In [ ]:
pip install transformers torch sentencepiece

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer


# Load pre-trained neural model
model_name = "vennify/t5-base-grammar-correction"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)


def neural_correct(sentence):

    # T5 expects a task prefix
    input_text = "grammar: " + sentence

    inputs = tokenizer(
        input_text,
        return_tensors="pt"
    )

    output = model.generate(
        **inputs,
        max_length=100
    )

    corrected = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return corrected


# Test
sentence = input("Enter sentence: ")

print("\nOriginal:")
print(sentence)

print("\nNeural Model Correction:")
print(neural_correct(sentence))

tokenizer_config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Enter sentence: she go to school yesterday

Original:
she go to school yesterday

Neural Model Correction:
She went to school yesterday.


Write a program

1. To compute unsmoothed unigrams and bigrams from a small corpora of your choice (Use only the train set) Add an option to compute the probability of any text entered by the user.
2. Add an option to generate random sentences using (i) Unigram, (ii) Bigrams
3. Compare the quality of your generated sentences.
4. Add an option to your program to compute the perplexity of a test set. Compare the perplexity of unigram and bigram models.

In [ ]:
import re
import random
import math
from collections import Counter


# ============================================================
# 1. TRAIN AND TEST CORPUS
# ============================================================

train_corpus = """
the cat sat on the mat
the dog sat on the rug
the cat ate the fish
the dog ate the food
the boy played with the ball
the girl played with the cat
the cat chased the mouse
the dog chased the cat
the boy ate the food
the girl read the book
"""

test_corpus = """
the cat sat on the rug
the dog ate the fish
the girl played with the ball
"""


# ============================================================
# 2. TOKENIZATION
# ============================================================

def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())


train_words = tokenize(train_corpus)
test_words = tokenize(test_corpus)

# Vocabulary from TRAIN ONLY
vocabulary = set(train_words)


# ============================================================
# 3. COMPUTE UNIGRAMS
# ============================================================

unigram_counts = Counter(train_words)

total_words = len(train_words)

unigram_probabilities = {}

for word in vocabulary:
    unigram_probabilities[word] = (
        unigram_counts[word] / total_words
    )


# ============================================================
# 4. COMPUTE BIGRAMS
# ============================================================

bigram_counts = Counter()

for i in range(len(train_words) - 1):
    bigram = (train_words[i], train_words[i + 1])
    bigram_counts[bigram] += 1


# Bigram probability:
# P(word2 | word1) = count(word1, word2) / count(word1)

bigram_probabilities = {}

for (word1, word2), count in bigram_counts.items():

    bigram_probabilities[(word1, word2)] = (
        count / unigram_counts[word1]
    )


# ============================================================
# DISPLAY COUNTS AND PROBABILITIES
# ============================================================

def show_models():

    print("\n========== UNIGRAM MODEL ==========")

    for word in sorted(vocabulary):
        print(
            f"{word:10} "
            f"Count = {unigram_counts[word]:2} "
            f"Probability = {unigram_probabilities[word]:.4f}"
        )

    print("\n========== BIGRAM MODEL ==========")

    for bigram, count in bigram_counts.items():

        probability = bigram_probabilities[bigram]

        print(
            f"{bigram} "
            f"Count = {count:2} "
            f"Probability = {probability:.4f}"
        )


# ============================================================
# 5. PROBABILITY OF USER TEXT
# ============================================================

def unigram_text_probability(text):

    words = tokenize(text)

    probability = 1.0

    for word in words:

        if word not in vocabulary:
            return 0

        probability *= unigram_probabilities[word]

    return probability


def bigram_text_probability(text):

    words = tokenize(text)

    if len(words) < 2:
        return 0

    probability = 1.0

    for i in range(len(words) - 1):

        bigram = (words[i], words[i + 1])

        if bigram not in bigram_probabilities:
            return 0

        probability *= bigram_probabilities[bigram]

    return probability


def calculate_text_probability():

    text = input("\nEnter a sentence: ")

    uni_prob = unigram_text_probability(text)
    bi_prob = bigram_text_probability(text)

    print("\nUnigram Probability:", uni_prob)
    print("Bigram Probability :", bi_prob)


# ============================================================
# 6. RANDOM SENTENCE USING UNIGRAM
# ============================================================

def generate_unigram_sentence(length=8):

    words = list(vocabulary)

    sentence = random.choices(
        words,
        weights=[unigram_counts[w] for w in words],
        k=length
    )

    return " ".join(sentence)


# ============================================================
# 7. RANDOM SENTENCE USING BIGRAM
# ============================================================

def generate_bigram_sentence(length=8):

    # Start with a random word
    current_word = random.choice(train_words)

    sentence = [current_word]

    for _ in range(length - 1):

        # Find possible next words
        candidates = []

        for (word1, word2) in bigram_counts:

            if word1 == current_word:
                candidates.append(word2)

        # If no next word exists, restart
        if not candidates:
            current_word = random.choice(train_words)
            sentence.append(current_word)
            continue

        # Choose based on bigram frequency
        next_word = random.choices(
            candidates,
            weights=[
                bigram_counts[(current_word, w)]
                for w in candidates
            ]
        )[0]

        sentence.append(next_word)

        current_word = next_word

    return " ".join(sentence)


def generate_sentences():

    print("\nUnigram Generated Sentence:")
    print(generate_unigram_sentence())

    print("\nBigram Generated Sentence:")
    print(generate_bigram_sentence())


# ============================================================
# 8. PERPLEXITY
# ============================================================

def unigram_perplexity(test_text):

    words = tokenize(test_text)

    log_probability = 0
    N = len(words)

    for word in words:

        if word not in vocabulary:
            return float("inf")

        probability = unigram_probabilities[word]

        log_probability += math.log(probability)

    perplexity = math.exp(
        -log_probability / N
    )

    return perplexity


def bigram_perplexity(test_text):

    words = tokenize(test_text)

    log_probability = 0
    N = len(words) - 1

    for i in range(len(words) - 1):

        bigram = (words[i], words[i + 1])

        if bigram not in bigram_probabilities:
            return float("inf")

        probability = bigram_probabilities[bigram]

        log_probability += math.log(probability)

    perplexity = math.exp(
        -log_probability / N
    )

    return perplexity


def calculate_perplexity():

    uni_pp = unigram_perplexity(test_corpus)
    bi_pp = bigram_perplexity(test_corpus)

    print("\n========== PERPLEXITY ==========")

    print("Unigram Perplexity:", uni_pp)
    print("Bigram Perplexity :", bi_pp)

    if uni_pp < bi_pp:
        print("\nUnigram model has lower perplexity.")

    elif bi_pp < uni_pp:
        print("\nBigram model has lower perplexity.")

    else:
        print("\nBoth models have equal perplexity.")


# ============================================================
# 9. MAIN MENU
# ============================================================

def main():

    while True:

        print("\n================================")
        print("       LANGUAGE MODEL")
        print("================================")
        print("1. Show unigram and bigram models")
        print("2. Compute probability of text")
        print("3. Generate random sentences")
        print("4. Compute test-set perplexity")
        print("5. Exit")

        choice = input("\nEnter your choice: ")

        if choice == "1":
            show_models()

        elif choice == "2":
            calculate_text_probability()

        elif choice == "3":
            generate_sentences()

        elif choice == "4":
            calculate_perplexity()

        elif choice == "5":
            print("Goodbye!")
            break

        else:
            print("Invalid choice.")


if __name__ == "__main__":
    main()


       LANGUAGE MODEL
1. Show unigram and bigram models
2. Compute probability of text
3. Generate random sentences
4. Compute test-set perplexity
5. Exit

Enter your choice: 1

========== UNIGRAM MODEL ==========
ate        Count =  3 Probability = 0.0556
ball       Count =  1 Probability = 0.0185
book       Count =  1 Probability = 0.0185
boy        Count =  2 Probability = 0.0370
cat        Count =  5 Probability = 0.0926
chased     Count =  2 Probability = 0.0370
dog        Count =  3 Probability = 0.0556
fish       Count =  1 Probability = 0.0185
food       Count =  2 Probability = 0.0370
girl       Count =  2 Probability = 0.0370
mat        Count =  1 Probability = 0.0185
mouse      Count =  1 Probability = 0.0185
on         Count =  2 Probability = 0.0370
played     Count =  2 Probability = 0.0370
read       Count =  1 Probability = 0.0185
rug        Count =  1 Probability = 0.0185
sat        Count =  2 Probability = 0.0370
the        Count = 20 Probability = 0.3704
with       

Implement a text classifier for sentiment analysis using the Naive Bayes theorem. Use Add-k smoothing to handle zero probabilities.

Compare the performance of your classifier for k values 0.25, 0.75, and 1.

In [ ]:
import re
from collections import Counter


# ============================================================
# 1. SMALL SENTIMENT DATASET
# ============================================================

train_data = [
    ("I love this movie", "positive"),
    ("This movie is excellent", "positive"),
    ("Amazing acting and great story", "positive"),
    ("I really enjoyed this film", "positive"),
    ("The movie was fantastic", "positive"),

    ("I hate this movie", "negative"),
    ("This movie is terrible", "negative"),
    ("Awful acting and boring story", "negative"),
    ("I really disliked this film", "negative"),
    ("The movie was horrible", "negative"),
]


test_data = [
    ("I love this film", "positive"),
    ("The movie was amazing", "positive"),
    ("I hate this film", "negative"),
    ("The movie was boring", "negative"),
]


# ============================================================
# 2. TOKENIZATION
# ============================================================

def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())


# ============================================================
# 3. TRAIN NAIVE BAYES
# ============================================================

positive_words = Counter()
negative_words = Counter()

positive_documents = 0
negative_documents = 0

for text, label in train_data:

    words = tokenize(text)

    if label == "positive":
        positive_words.update(words)
        positive_documents += 1

    else:
        negative_words.update(words)
        negative_documents += 1


# Vocabulary
vocabulary = set(positive_words) | set(negative_words)

vocab_size = len(vocabulary)

total_positive_words = sum(positive_words.values())
total_negative_words = sum(negative_words.values())

total_documents = len(train_data)


# ============================================================
# 4. NAIVE BAYES CLASSIFIER
# ============================================================

def classify(text, k):

    words = tokenize(text)

    # Prior probabilities
    p_positive = positive_documents / total_documents
    p_negative = negative_documents / total_documents

    # Use log probabilities to avoid very small numbers
    log_positive = 0
    log_negative = 0

    for word in words:

        # Add-k smoothing
        p_word_positive = (
            positive_words[word] + k
        ) / (
            total_positive_words + k * vocab_size
        )

        p_word_negative = (
            negative_words[word] + k
        ) / (
            total_negative_words + k * vocab_size
        )

        log_positive += math.log(p_word_positive)
        log_negative += math.log(p_word_negative)

    # Add class prior
    log_positive += math.log(p_positive)
    log_negative += math.log(p_negative)

    if log_positive > log_negative:
        return "positive"

    else:
        return "negative"


# ============================================================
# 5. EVALUATE CLASSIFIER
# ============================================================

def evaluate(k):

    correct = 0

    print(f"\n========== k = {k} ==========")

    for text, actual in test_data:

        predicted = classify(text, k)

        print(
            f"Text: {text}\n"
            f"Actual: {actual}\n"
            f"Predicted: {predicted}\n"
        )

        if predicted == actual:
            correct += 1

    accuracy = correct / len(test_data)

    print("Accuracy:", accuracy * 100, "%")

    return accuracy


# ============================================================
# 6. COMPARE DIFFERENT k VALUES
# ============================================================

import math

results = {}

for k in [0.25, 0.75, 1]:

    accuracy = evaluate(k)

    results[k] = accuracy


print("\n========== FINAL COMPARISON ==========")

for k, accuracy in results.items():

    print(
        f"k = {k:<4} "
        f"Accuracy = {accuracy * 100:.2f}%"
    )


========== k = 0.25 ==========
Text: I love this film
Actual: positive
Predicted: positive

Text: The movie was amazing
Actual: positive
Predicted: positive

Text: I hate this film
Actual: negative
Predicted: negative

Text: The movie was boring
Actual: negative
Predicted: negative

Accuracy: 100.0 %

========== k = 0.75 ==========
Text: I love this film
Actual: positive
Predicted: positive

Text: The movie was amazing
Actual: positive
Predicted: positive

Text: I hate this film
Actual: negative
Predicted: negative

Text: The movie was boring
Actual: negative
Predicted: negative

Accuracy: 100.0 %

========== k = 1 ==========
Text: I love this film
Actual: positive
Predicted: positive

Text: The movie was amazing
Actual: positive
Predicted: positive

Text: I hate this film
Actual: negative
Predicted: negative

Text: The movie was boring
Actual: negative
Predicted: negative

Accuracy: 100.0 %

========== FINAL COMPARISON ==========
k = 0.25 Accuracy = 100.00%
k = 0.75 Accuracy = 100.00